# 📐 Regresión Linear - Implementación desde CERO

## Objetivos
- Entender la matemática detrás de regresión linear
- Implementar regresión linear desde cero usando NumPy
- Implementar Gradient Descent
- Visualizar el proceso de aprendizaje
- Evaluar el modelo

---

## Tabla de Contenidos
- [1 - Paquetes](#1)
- [2 - Conceptos Principales](#2)
- [3 - Ejercicios Prácticos](#3)
- [4 - Resumen](#4)

In [ ]:
# ==========================================
# CONFIGURACIÓN DEL ENTORNO
# ==========================================
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Agregar el directorio raíz al path de manera robusta
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verificar que las utilidades se pueden importar
try:
    from utils.test_utils import check_answer, check_model, print_test_summary
    from utils.plot_utils import plot_regression_results, plot_learning_curve
    print("✅ Entorno configurado correctamente")
    print(f"📁 Raíz del proyecto: {project_root}")
except ImportError as e:
    print("❌ Error al importar utilidades")
    print("\n💡 Soluciones:")
    print("   1. Ejecuta 'pip install -e .' desde la raíz del proyecto")
    print("   2. O inicia Jupyter desde la raíz: cd ML-FROM-ZERO-PYTHON && jupyter notebook")
    print(f"\n🔍 Error detallado: {e}")
    raise

## 1. Teoría de Regresión Linear

### El Modelo

La regresión linear busca encontrar una relación linear entre las features $X$ y el target $y$:

$$\hat{y} = w_0 + w_1x_1 + w_2x_2 + ... + w_nx_n$$

O en forma vectorial:
$$\hat{y} = w^T X + b$$

Donde:
- $\hat{y}$ = predicción
- $w$ = pesos (weights)
- $b$ = sesgo (bias/intercept)
- $X$ = features

### Función de Costo (Loss Function)

Usamos **Mean Squared Error (MSE)**:

$$J(w, b) = \frac{1}{2m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})^2$$

Nuestro objetivo: **Minimizar $J(w, b)$**

### Gradient Descent

Actualizamos los parámetros iterativamente:

$$w := w - \alpha \frac{\partial J}{\partial w}$$
$$b := b - \alpha \frac{\partial J}{\partial b}$$

Donde $\alpha$ es el learning rate.

### Las Derivadas

$$\frac{\partial J}{\partial w} = \frac{1}{m} X^T (\hat{y} - y)$$
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum (\hat{y} - y)$$

## 2. Implementación desde CERO

In [ ]:
class RegresionLinear:
    """
    Regresión Linear implementada desde cero con Gradient Descent.
    
    Parámetros:
    -----------
    learning_rate : float
        Tasa de aprendizaje (alpha)
    n_iterations : int
        Número de iteraciones para gradient descent
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.loss_history = []
    
    def fit(self, X, y):
        """
        Entrena el modelo usando Gradient Descent.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features de entrenamiento
        y : array-like, shape (n_samples,)
            Target de entrenamiento
        """
        # Convertir a numpy arrays
        X = np.array(X)
        y = np.array(y)
        
        # Si X es 1D, convertir a 2D
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        # Número de muestras y features
        n_samples, n_features = X.shape
        
        # Inicializar parámetros
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient Descent
        for i in range(self.n_iterations):
            # Predicción
            y_pred = self.predict(X)
            
            # Calcular gradientes
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Actualizar parámetros
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            # Guardar loss
            loss = self._mse(y, y_pred)
            self.loss_history.append(loss)
            
            # Imprimir progreso cada 100 iteraciones
            if (i + 1) % 100 == 0:
                print(f"Iteración {i+1}/{self.n_iterations}, Loss: {loss:.4f}")
        
        return self
    
    def predict(self, X):
        """
        Hace predicciones.
        
        Parámetros:
        -----------
        X : array-like, shape (n_samples, n_features)
            Features para predecir
        
        Returns:
        --------
        array, shape (n_samples,)
            Predicciones
        """
        X = np.array(X)
        if len(X.shape) == 1:
            X = X.reshape(-1, 1)
        
        return np.dot(X, self.weights) + self.bias
    
    def _mse(self, y_true, y_pred):
        """Calcula Mean Squared Error"""
        return np.mean((y_true - y_pred) ** 2)
    
    def score(self, X, y):
        """
        Calcula el R² score.
        
        Returns:
        --------
        float
            R² score
        """
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

## 3. Ejemplo Simple: Una Variable

In [ ]:
# Generar datos sintéticos
np.random.seed(42)
X = 2 * np.random.rand(100)
y = 4 + 3 * X + np.random.randn(100)  # y = 4 + 3x + ruido

print(f"Datos generados: {len(X)} muestras")
print(f"Relación real: y = 4 + 3x + ruido")

# Visualizar datos
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6)
plt.xlabel('X')
plt.ylabel('y')
plt.title('Datos de Entrenamiento')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Entrenar el modelo
modelo = RegresionLinear(learning_rate=0.1, n_iterations=1000)
modelo.fit(X, y)

print(f"\nParámetros aprendidos:")
print(f"Peso (w): {modelo.weights[0]:.4f}")
print(f"Bias (b): {modelo.bias:.4f}")
print(f"\nEcuación aprendida: y = {modelo.bias:.4f} + {modelo.weights[0]:.4f}x")
print(f"Ecuación real:      y = 4.0000 + 3.0000x")

In [ ]:
# Visualizar resultados
plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, label='Datos reales')

# Línea de predicción
X_plot = np.linspace(X.min(), X.max(), 100)
y_plot = modelo.predict(X_plot)
plt.plot(X_plot, y_plot, color='red', linewidth=2, label='Predicción')

plt.xlabel('X')
plt.ylabel('y')
plt.title('Regresión Linear - Ajuste del Modelo')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Visualizar curva de aprendizaje (cómo baja el loss)
plt.figure(figsize=(10, 6))
plt.plot(modelo.loss_history, linewidth=2)
plt.xlabel('Iteración')
plt.ylabel('Loss (MSE)')
plt.title('Curva de Aprendizaje - Convergencia de Gradient Descent')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Loss inicial: {modelo.loss_history[0]:.4f}")
print(f"Loss final: {modelo.loss_history[-1]:.4f}")
print(f"Reducción: {(1 - modelo.loss_history[-1]/modelo.loss_history[0]) * 100:.2f}%")

## 4. Ejemplo con Múltiples Variables

In [ ]:
# Generar datos con 3 features
np.random.seed(42)
n_samples = 200
X_multi = np.random.rand(n_samples, 3)

# Relación real: y = 5 + 2*x1 - 3*x2 + 4*x3 + ruido
y_multi = 5 + 2*X_multi[:, 0] - 3*X_multi[:, 1] + 4*X_multi[:, 2] + np.random.randn(n_samples) * 0.5

print(f"Datos generados: {n_samples} muestras, 3 features")
print(f"Relación real: y = 5 + 2*x1 - 3*x2 + 4*x3 + ruido")

In [ ]:
# Train/Test split manual
def train_test_split(X, y, test_size=0.2, random_state=None):
    if random_state:
        np.random.seed(random_state)
    
    n = len(X)
    indices = np.random.permutation(n)
    test_size_n = int(n * test_size)
    
    test_idx = indices[:test_size_n]
    train_idx = indices[test_size_n:]
    
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

print(f"Train set: {len(X_train)} muestras")
print(f"Test set: {len(X_test)} muestras")

In [ ]:
# Entrenar modelo
modelo_multi = RegresionLinear(learning_rate=0.1, n_iterations=1000)
modelo_multi.fit(X_train, y_train)

print(f"\nParámetros aprendidos:")
print(f"Bias: {modelo_multi.bias:.4f}")
print(f"Pesos: {modelo_multi.weights}")
print(f"\nParámetros reales:")
print(f"Bias: 5.0000")
print(f"Pesos: [2.0000, -3.0000, 4.0000]")

In [ ]:
# Evaluar en train y test
train_score = modelo_multi.score(X_train, y_train)
test_score = modelo_multi.score(X_test, y_test)

print(f"R² en Training: {train_score:.4f}")
print(f"R² en Test: {test_score:.4f}")

if abs(train_score - test_score) < 0.1:
    print("\n✅ El modelo generaliza bien (no hay overfitting)")
else:
    print("\n⚠️  Posible overfitting")

## 5. Normalización de Features

Cuando las features tienen escalas muy diferentes, normalizar ayuda a que gradient descent converja más rápido.

In [ ]:
# Datos con escalas muy diferentes
np.random.seed(42)
X_sin_normalizar = np.random.rand(100, 2)
X_sin_normalizar[:, 0] *= 1000  # Primera feature: 0-1000
X_sin_normalizar[:, 1] *= 1     # Segunda feature: 0-1

y_escala = 5 + 2*X_sin_normalizar[:, 0] + 3*X_sin_normalizar[:, 1] + np.random.randn(100)*10

print("Estadísticas de features SIN normalizar:")
print(f"Feature 1 - min: {X_sin_normalizar[:, 0].min():.2f}, max: {X_sin_normalizar[:, 0].max():.2f}")
print(f"Feature 2 - min: {X_sin_normalizar[:, 1].min():.2f}, max: {X_sin_normalizar[:, 1].max():.2f}")

In [ ]:
def normalizar(X):
    """Normalización: (X - media) / std"""
    return (X - np.mean(X, axis=0)) / np.std(X, axis=0)

X_normalizado = normalizar(X_sin_normalizar)

print("Estadísticas de features NORMALIZADAS:")
print(f"Feature 1 - media: {X_normalizado[:, 0].mean():.2f}, std: {X_normalizado[:, 0].std():.2f}")
print(f"Feature 2 - media: {X_normalizado[:, 1].mean():.2f}, std: {X_normalizado[:, 1].std():.2f}")

In [ ]:
# Comparar convergencia con y sin normalización
modelo_sin_norm = RegresionLinear(learning_rate=0.0001, n_iterations=1000)
modelo_sin_norm.fit(X_sin_normalizar, y_escala)

modelo_con_norm = RegresionLinear(learning_rate=0.01, n_iterations=1000)
modelo_con_norm.fit(X_normalizado, y_escala)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(modelo_sin_norm.loss_history)
axes[0].set_title('SIN Normalización\n(learning_rate=0.0001)')
axes[0].set_xlabel('Iteración')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(modelo_con_norm.loss_history)
axes[1].set_title('CON Normalización\n(learning_rate=0.01)')
axes[1].set_xlabel('Iteración')
axes[1].set_ylabel('Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Loss final SIN normalización: {modelo_sin_norm.loss_history[-1]:.4f}")
print(f"Loss final CON normalización: {modelo_con_norm.loss_history[-1]:.4f}")

## 🎯 Ejercicio 1: Implementa tu propia Regresión Linear

Implementa una clase similar pero más simple.

In [ ]:
# TU CÓDIGO AQUÍ
class MiRegresionLinear:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
    
    def fit(self, X, y):
        # Implementa aquí el entrenamiento
        pass
    
    def predict(self, X):
        # Implementa aquí la predicción
        pass

# Prueba tu implementación
# mi_modelo = MiRegresionLinear()
# mi_modelo.fit(X, y)

## 🎯 Ejercicio 2: Predicción de Precios de Casas

Crea datos sintéticos de casas y entrena un modelo para predecir precios.

In [ ]:
# TU CÓDIGO AQUÍ
# Features: tamaño (m²), número de habitaciones, antigüedad (años)
# Target: precio (miles de €)

# 1. Genera datos sintéticos
# 2. Divide en train/test
# 3. Normaliza features
# 4. Entrena modelo
# 5. Evalúa con R²
# 6. Haz predicciones

print("Implementa el ejercicio aquí")

## 🎓 Resumen

Has aprendido:

1. ✅ La matemática detrás de regresión linear
2. ✅ Implementación desde cero con Gradient Descent
3. ✅ Cómo funciona la optimización
4. ✅ Regresión con múltiples variables
5. ✅ Importancia de la normalización
6. ✅ Evaluación con R²
7. ✅ Detección de overfitting

### Próximo: Regresión Logística para Clasificación